# Mini-Projet : Spark avec Docker

**Binôme :** Mazilda ZEHRAOUI et [nom du binôme]

Analyse des ventes d'une chaîne de magasins (`purchases.txt`) avec Apache Spark : RDD, DataFrame API, Spark SQL et un mini-pipeline ETL vers Parquet.

| Étape | Contenu |
|---|---|
| 1 | Environnement Docker et SparkSession |
| 2 | Chargement et exploration |
| 3 | Traitement avec les RDD |
| 4 | DataFrame API |
| 5 | Spark SQL |
| 6 | Pipeline ETL et Parquet |

# Étape 1 : Préparation de l'environnement Docker

Le conteneur est lancé avec `docker compose up -d`, puis Jupyter est ouvert sur `http://localhost:8888`. On initialise ensuite la session Spark.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MiniProjet_Spark").master("local[2]").getOrCreate()
print("Master =", spark.sparkContext.master)

### 1. Quel est le rôle de Docker dans ce mini-projet ?
Docker nous permet de lancer Spark et Jupyter dans un conteneur, sans avoir à installer toutes les dépendances sur nos ordinateurs.

### 2. Pourquoi utilisons-nous une image Docker contenant PySpark ?
L'image contient déjà Python, PySpark et Jupyter. On peut donc directement travailler avec Spark sans devoir tout configurer nous-mêmes.

### 3. Quelle est la différence entre un conteneur Docker et une machine virtuelle ?
Un conteneur Docker partage le noyau du système sur lequel il tourne, alors qu'une machine virtuelle possède son propre système d'exploitation et son propre noyau. Docker est donc généralement plus léger et plus rapide à démarrer.

### 4. À quoi sert le fichier docker-compose.yml ?
Il sert à configurer et lancer notre conteneur. Dans notre cas, il précise l'image PySpark à utiliser, le port pour accéder à Jupyter et le dossier partagé entre notre ordinateur et le conteneur.

### 5. Que signifie le mode d'exécution local[2] de Spark ?
Cela signifie que Spark tourne sur une seule machine avec 2 threads, ce qui lui permet d'exécuter jusqu'à 2 tâches en parallèle.

### 6. Quel est le rôle de SparkSession ?
C'est le point d'entrée de Spark. On l'utilise pour travailler avec les DataFrames, charger des données et lancer des requêtes SQL.


# Étape 2 : Chargement et exploration des données

Le fichier n'a pas d'en-tête et ses champs sont séparés par des tabulations. Remarque : le sujet parle d'une colonne « Produit ». Dans le fichier, ce champ contient une catégorie de produits (ex. *Music*, *Pet Supplies*), on l'a donc nommée `Categorie`.

In [ ]:
import os
from pyspark.sql.functions import col
print(os.listdir("/home/jovyan/work"))

In [ ]:
with open("/home/jovyan/work/purchases.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().rstrip())

In [ ]:
# 1. Charger les données dans un DataFrame
df = spark.read.csv(
    "/home/jovyan/work/purchases.txt",
    sep="\t",
    header=False
)

In [ ]:
# 2. Définir les noms des colonnes
print(df.columns)
df = df.toDF("Date", "Heure", "Magasin", "Categorie", "Cout", "Paiement")
print(df.columns)

In [ ]:
# 3. Afficher les 10 premières lignes
df.show(10)

In [ ]:
# 4. Afficher le schéma du DataFrame
df.printSchema()

In [ ]:
# 5. Vérifier et convertir Cout en numerique
print(f"Cout type : {df.schema["Cout"].dataType}")

df = df.withColumn("Cout", col("Cout").cast("double"))

df.printSchema()


In [ ]:
# 6. Nombre total de transactions
nb_transactions = df.count()

print("Nombre total de transactions :", nb_transactions)

**Interprétation :** à la lecture, toutes les colonnes sont de type `string`. On convertit `Cout` en `double` pour pouvoir faire des filtres et des calculs (somme, moyenne).

# Étape 3 : Traitement avec les RDD

On repart du fichier brut `purchases.txt` avec l'API bas niveau de Spark (RDD). Chaque ligne contient 6 champs séparés par une tabulation : Date, Heure, Magasin, Catégorie, Coût, Paiement.

In [ ]:
sc = spark.sparkContext

# Création du RDD à partir du fichier texte
rdd = sc.textFile("/home/jovyan/work/purchases.txt")

# 1. Afficher les cinq premiers éléments
for ligne in rdd.take(5):
    print(ligne)

Chaque élément du RDD est pour l'instant une simple chaîne de caractères (une ligne du fichier).

In [ ]:
# 2. map() : découper chaque ligne en liste de champs
rdd_champs = rdd.map(lambda ligne: ligne.split("\t"))

# On écarte les éventuelles lignes mal formées (≠ 6 champs)
rdd_champs = rdd_champs.filter(lambda champs: len(champs) == 6)

rdd_champs.take(5)

In [ ]:
# 3. Extraire les couples (Magasin, Cout), avec Cout converti en nombre
rdd_magasin_cout = rdd_champs.map(lambda champs: (champs[2], float(champs[4])))

rdd_magasin_cout.take(5)

In [ ]:
# 4. filter() : garder les transactions dont le montant est > 100
rdd_sup_100 = rdd_magasin_cout.filter(lambda x: x[1] > 100)

print("Transactions > 100 :", rdd_sup_100.count())
rdd_sup_100.take(5)

In [ ]:
# 5. reduceByKey() : montant total des ventes par magasin
rdd_ca_magasin = rdd_magasin_cout.reduceByKey(lambda a, b: a + b)

print("Nombre de magasins :", rdd_ca_magasin.count())
# Affichage des 10 plus gros CA (tri décroissant)
for magasin, total in rdd_ca_magasin.sortBy(lambda x: x[1], ascending=False).take(10):
    print(f"{magasin:<20} {total:,.2f}")

In [ ]:
# Variante : total par magasin uniquement sur les ventes > 100
rdd_ca_sup_100 = rdd_sup_100.reduceByKey(lambda a, b: a + b)

rdd_ca_sup_100.sortBy(lambda x: x[1], ascending=False).take(5)

**Interprétation :** on obtient un chiffre d'affaires par magasin. Ces totaux sont les mêmes que ceux calculés avec la DataFrame API et Spark SQL (étapes 4 et 5), mais le code RDD est plus long : il faut découper les lignes et convertir les types à la main.

### Questions

**1. Différence entre `map()` et `filter()` ?**  
`map()` applique une fonction à chaque élément et renvoie un élément transformé pour chacun (même nombre d'éléments). `filter()` ne modifie pas les éléments : il garde seulement ceux qui vérifient une condition (moins d'éléments ou autant).

**2. Rôle de `reduceByKey()` ?**  
Sur un RDD de couples (clé, valeur), il regroupe les valeurs qui ont la même clé et les combine avec une fonction (ici l'addition). Il fait d'abord une réduction locale dans chaque partition, puis combine les résultats, ce qui limite les données échangées entre les partitions (shuffle).

**3. Différence entre une transformation et une action ?**  
Une transformation (`map`, `filter`, `reduceByKey`…) crée un nouveau RDD et est *paresseuse* (lazy) : Spark l'enregistre sans l'exécuter. Une action (`take`, `count`, `collect`…) déclenche réellement le calcul et renvoie un résultat au driver.

# Étape 4 : Analyse avec la DataFrame API

On utilise `select()`, `filter()`, `orderBy()`, `groupBy()` et `agg()` sur la DataFrame de l'étape 2.

In [ ]:
from pyspark.sql.functions import sum, avg

# 1. Afficher uniquement les colonnes Magasin, Categorie et Cout
df.select("Magasin", "Categorie", "Cout").show(10)

In [ ]:
# 2. Filtrer les ventes dont le montant est supérieur à 100
df.filter(col("Cout") > 100).show(10)

In [ ]:
# 3. Trier ces ventes par montant décroissant
df.filter(col("Cout") > 100).orderBy(col("Cout").desc()).show(10)

In [ ]:
# 4. Calculer le chiffre d'affaires total par magasin
df_ca_magasin = df.groupBy("Magasin").agg(sum("Cout").alias("CA_total"))

df_ca_magasin.show()

In [ ]:
# 5. Identifier le magasin ayant le chiffre d'affaires le plus élevé
df_ca_magasin.orderBy(col("CA_total").desc()).show(1)

In [ ]:
# 6. Calculer le montant moyen des transactions par catégorie de produits
df.groupBy("Categorie").agg(avg("Cout").alias("Montant_moyen")).show()

# Étape 5 : Analyse avec Spark SQL

On crée une vue temporaire `ventes` pour interroger la DataFrame en SQL.

In [ ]:
# 1. Créer une vue temporaire nommée ventes
df.createOrReplaceTempView("ventes")

In [ ]:
# 2. Afficher les ventes dont le montant dépasse 100
spark.sql("""
    SELECT *
    FROM ventes
    WHERE Cout > 100
""").show(10)

In [ ]:
# 3. Calculer le chiffre d'affaires total par magasin
df_ca_sql = spark.sql("""
    SELECT
        Magasin,
        SUM(Cout) AS CA_total
    FROM ventes
    GROUP BY Magasin
""")

df_ca_sql.show()

In [ ]:
# 4. Afficher les trois magasins ayant le chiffre d'affaires le plus élevé
spark.sql("""
    SELECT
        Magasin,
        SUM(Cout) AS CA_total
    FROM ventes
    GROUP BY Magasin
    ORDER BY CA_total DESC
    LIMIT 3
""").show()

### 5. Comparaison entre Spark SQL et DataFrame API

Les deux méthodes donnent les mêmes résultats. Avec la DataFrame API, on utilise des fonctions Python comme `filter()` ou `groupBy()`, alors qu'avec Spark SQL, on écrit directement des requêtes SQL.

Par exemple, pour afficher les ventes supérieures à 100 :

**DataFrame API :**
```python
df.filter(col("Cout") > 100).show()
```

**Spark SQL :**
```sql
SELECT * FROM ventes WHERE Cout > 100
```

C'est surtout la syntaxe qui change donc, puisque les traitements sont exécutés par le même moteur Spark.


# Étape 6 : Mini-pipeline ETL avec PySpark

### Sauvegarde des résultats

In [ ]:
# 1. Récupérer la DataFrame du chiffre d'affaires par magasin (étape 4)
df_resultat = df_ca_magasin

df_resultat.orderBy("Magasin").show(10)

In [ ]:
# 2. Enregistrer au format Parquet (mode overwrite pour pouvoir réexécuter)
output_path = "/home/jovyan/work/output/vente_par_magasin"

df_resultat.write.mode("overwrite").parquet(output_path)

In [ ]:
# 3. Vérifier que les fichiers ont bien été créés
import os
print(os.listdir(output_path))

Le dossier contient un ou plusieurs fichiers `part-*.parquet` (les données) et un fichier `_SUCCESS` qui indique que l'écriture s'est bien terminée.

### Vérification des données enregistrées

In [ ]:
# 1. Relire les données Parquet dans une nouvelle DataFrame
df_verification = spark.read.parquet(output_path)

# 2. Afficher les premières lignes
df_verification.orderBy("Magasin").show(10)

In [ ]:
# 3. Vérifier le schéma
df_verification.printSchema()

In [ ]:
# 4. Comparer avec les résultats d'avant l'enregistrement
print("Lignes avant :", df_resultat.count())
print("Lignes après :", df_verification.count())
print("Même schéma  :", df_resultat.schema == df_verification.schema)

# Différences dans les deux sens (0 = données identiques)
diff_1 = df_resultat.exceptAll(df_verification).count()
diff_2 = df_verification.exceptAll(df_resultat).count()
print("Lignes différentes :", diff_1 + diff_2)

**Interprétation :** même nombre de lignes, même schéma (Magasin : string, CA_total : double) et aucune ligne différente. Le format Parquet a conservé les données et leurs types. L'ordre d'affichage peut varier, c'est pourquoi on trie par magasin avant de comparer visuellement.

### Questions

**1. Phases d'un pipeline ETL, et où on les trouve dans le projet :**
- **Extract** (extraire) : lecture de `purchases.txt` (étape 2 en DataFrame, étape 3 en RDD).
- **Transform** (transformer) : nommage des colonnes, conversion de `Cout` en double, filtres, agrégations par magasin et par catégorie (étapes 3, 4 et 5).
- **Load** (charger) : écriture du CA par magasin au format Parquet (étape 6).

**2. Pourquoi utiliser Parquet en Big Data ?**  
C'est un format colonnaire, compressé, qui garde le schéma. Spark ne lit que les colonnes utiles, les fichiers sont plus petits et les lectures plus rapides. Il se découpe aussi bien pour un traitement en parallèle.

**3. Différence entre CSV et Parquet ?**  
CSV : texte lisible, stocké ligne par ligne, sans types (tout est lu comme du texte, il faut ré-inférer ou caster), sans compression. Parquet : binaire, stocké par colonnes, compressé, avec le schéma et les types inclus.

**4. Intérêt du mode `overwrite` ?**  
Il remplace le contenu du dossier s'il existe déjà. On peut donc relancer le notebook sans erreur (le mode par défaut échoue si le dossier existe) et sans accumuler d'anciens résultats.

**5. Différence entre `show()` et l'enregistrement en Parquet ?**  
`show()` affiche seulement quelques lignes à l'écran, et rien n'est conservé après l'arrêt de la session. Enregistrer en Parquet écrit toutes les données sur disque : elles sont persistantes et réutilisables par d'autres traitements.

# Conclusion

Les trois approches (RDD, DataFrame API, Spark SQL) donnent le même chiffre d'affaires par magasin. Les RDD demandent plus de code (découpage et conversion à la main). La DataFrame API et Spark SQL sont plus courts et plus lisibles, et ils passent par l'optimiseur de Spark (Catalyst). Le résultat final est enregistré en Parquet, puis relu à l'identique.